# Raw-EDF seizure likelihood and time-to-onset network

This notebook replaces the broken random-data regression demo. It trains two small neural networks on filter-aware 6-second EEG windows generated from the Siena EDF files:

- `seizure_likelihood`: an experimental 0–1 score for an annotated seizure beginning within the next 60 seconds.
- `predicted_seconds_to_seizure`: a conditional timing estimate, returned only when the score crosses a threshold selected on held-out patients.

**Research use only.** The score is not a clinically calibrated probability, and this notebook must not be used for medical decisions or patient monitoring.

In [ ]:
from pathlib import Path
import sys
import json

CANDIDATE_SCRIPT_DIRS = [
    Path.cwd(),
    Path.cwd() / "final_project" / "scripts",
    Path.cwd() / "26-the-optimizers-analysis" / "final_project" / "scripts",
]
SCRIPT_DIR = next(
    (path for path in CANDIDATE_SCRIPT_DIRS if (path / "neural_network_pipeline.py").exists()),
    None,
)
if SCRIPT_DIR is None:
    raise FileNotFoundError("Could not locate final_project/scripts/neural_network_pipeline.py")
sys.path.insert(0, str(SCRIPT_DIR.resolve()))

from neural_network_pipeline import (
    DEFAULT_FEATURE_TABLE,
    ModelConfig,
    load_training_windows,
    predict_edf_window,
    train_models,
)

PROJECT_DIR = SCRIPT_DIR.parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
print(f"Project: {PROJECT_DIR.resolve()}")

## Train and patient-holdout validate

`all_window_features.csv` is the compact feature table previously extracted from the raw EDF files by `analyze_final60s_bandpower.py`. The split is by patient, not by neighboring windows, which reduces temporal and patient leakage.

In [ ]:
windows = load_training_windows(DEFAULT_FEATURE_TABLE)
fitted = train_models(windows, ModelConfig())
print(json.dumps(fitted["metrics"], indent=2))

## Score one raw EDF window

Set `EDF_PATH` and `WINDOW_START_SECONDS`. The function reads only the requested 6 seconds from the EDF, performs the same artifact checks and filter-aware band-power extraction used during training, and returns the score plus a conditional time estimate.

In [ ]:
EDF_PATH = RAW_DIR / "PN00" / "PN00-1.edf"
WINDOW_START_SECONDS = 1083  # Example: 60 seconds before the annotated onset at 1143 s

prediction = predict_edf_window(fitted, EDF_PATH, WINDOW_START_SECONDS)
print(f"Seizure likelihood (next 60 s): {prediction['seizure_likelihood']:.3f}")
print(f"Warning threshold: {prediction['warning_threshold']:.3f}")
print(f"Classifier validated: {prediction['classifier_validated']}")
print(f"Timing model validated: {prediction['timing_validated']}")
print(f"Predicted seconds to seizure: {prediction['predicted_seconds_to_seizure']}")
print(prediction["interpretation"])

### Interpretation guardrails

- The 0–1 output is a model score, not a clinically validated probability.
- A timing value is intentionally suppressed below the warning threshold; otherwise every ordinary EEG window would receive a misleading countdown.
- Validation holds out whole patients, but the dataset is small (14 patients, 47 seizures). Report the displayed holdout metrics and do not generalize them to clinical use.